## 1. Overview of products and their unique attributes
The purpose for this analysis, is to do a spectral clustering, simmilar to 5_spectral.ipynb. However, we focus this analysis on whether the clusters are simmilar within a each product type. 
Below, theres a table with of products that counts products and unique features per type, to show which types have enough features to form meaningful clusters.

In [68]:
import pandas as pd

products = pd.read_json("data/products_en.json")

summary = (
    products.groupby("product_type")
    .agg(products=("product_id", "count"),
         unique_attributes=("titles", lambda s: len(set(a for lst in s for a in lst))))
    .sort_values("unique_attributes", ascending=False)
)
summary

,products,unique_attributes
product_type,,
Cycling Wear Tops,42,93
Shoe Covers,21,70
Headwear,24,65
Cycling Wear Bottoms,31,62
Gloves Cycling Winter,15,58
Socks,25,56
Base Layers,21,40
Gloves Cycling Summer Long,6,20
Gloves Cycling Summer Short,7,20


## 2. Load the similarity matrix, filter to one product type, set parameters

`similarity.npz` was built once on the full catalog, from Qwen embeddings (50%), TF-IDF of the descriptions (25%) and fuzzy title matching (25%), with a penalty for conflicting concepts. For a product type, we **slice** this matrix to the rows and columns of that type's feature names.

Note: a feature's similarity values still reflect the whole catalog. The TF-IDF word weights come from all descriptions, and a feature's description list includes every product that uses it, not only those of the chosen type.

Steps:
1. Load the full matrix and feature names.
2. Keep only the feature names used by products of `CATEGORY`.
3. Check that the filtered matrix is square and symmetric.
4. Set the parameters that depend on the number of features `n`. This must happen after filtering. The full-run values (15 neighbours, 20 to 80 clusters) are too large for about 100 features, so they are scaled down: `K_NEIGHBORS = 8`, `K_MIN = 3`, `K_MAX = n // 5`.

In [69]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.cluster import KMeans
from scipy.sparse.csgraph import connected_components

SEED = 0
CATEGORY = "Cycling Wear Bottoms"   # the only line you change: "Headwear", "Shoe Covers", ...

# Load the full similarity matrix
with np.load("data/similarity.npz") as data:
    S = data["similarity"]
    attributes = data["titles"].tolist()
attributes_full = list(attributes)

# Filter to attributes used by products of this category
cat_df = products[products["product_type"] == CATEGORY]
category_attrs = set(cat_df["titles"].explode())

missing = category_attrs - set(attributes_full)
print(f"Titles in products but not in similarity.npz: {len(missing)}", list(missing)[:5])

keep = [i for i, a in enumerate(attributes_full) if a in category_attrs]
S = S[np.ix_(keep, keep)]
attributes = [attributes_full[i] for i in keep]
n = len(attributes)

# Checks on the filtered matrix
assert S.shape == (n, n)
assert np.allclose(S, S.T), "Similarity matrix must be symmetric"
print(f"{CATEGORY}: {len(cat_df)} products, {n} attributes")

# Settings that depend on n (must come after the filter)
K_NEIGHBORS = 60
K_MIN = 3
K_MAX = n // 5
N_EIGEN = K_MAX + 1

print(f"K_NEIGHBORS={K_NEIGHBORS}, K_MIN={K_MIN}, K_MAX={K_MAX}")

Titles in products but not in similarity.npz: 0 []
Cycling Wear Bottoms: 31 products, 62 attributes
K_NEIGHBORS=60, K_MIN=3, K_MAX=12


## 3. Build the graph, compute the Laplacian, estimate the number of clusters

1. **Affinity graph:** each feature is linked to its most similar features. An edge is kept if it is in the top `K_NEIGHBORS` of either endpoint, which keeps the graph symmetric.
2. **Normalized Laplacian:** L = I - D^(-1/2) W D^(-1/2). Rescaling by degree stops highly connected features from dominating.
3. **Eigenvectors:** those with the smallest eigenvalues describe how the graph splits into groups.
4. **Eigengap heuristic:** the number of clusters is taken at the largest jump between consecutive eigenvalues, searched between `K_MIN` and `K_MAX`.
5. **Connectivity check:** each disconnected component adds an eigenvalue of 0 and distorts the eigengap. If there is more than one component, `K_NEIGHBORS` is increased.

In [70]:
def build_affinity(matrix, k_neighbors):
    affinity = matrix.copy()
    np.fill_diagonal(affinity, 0.0)
    threshold = np.sort(affinity, axis=1)[:, -k_neighbors][:, None]
    return np.where((affinity >= threshold) | (affinity >= threshold.T), affinity, 0.0)


def spectral_embedding(affinity, n_eigen):
    degree = affinity.sum(axis=1)
    inv_sqrt = 1 / np.sqrt(np.maximum(degree, 1e-12))
    L = np.eye(len(affinity)) - affinity * np.outer(inv_sqrt, inv_sqrt)
    eigenvalues, eigenvectors = np.linalg.eigh(L)
    return eigenvalues[:n_eigen], eigenvectors[:, :n_eigen]


affinity = build_affinity(S, K_NEIGHBORS)
eigenvalues, eigenvectors = spectral_embedding(affinity, N_EIGEN)

gaps = np.diff(eigenvalues)
cluster_count = int(np.argmax(gaps[K_MIN - 1:K_MAX]) + K_MIN)

n_comp, _ = connected_components(affinity > 0, directed=False)
print(f"Edges kept: {np.count_nonzero(np.triu(affinity))}")
print(f"Connected components: {n_comp}")
print(f"Eigengap suggests {cluster_count} clusters")

Edges kept: 1578
Connected components: 1
Eigengap suggests 3 clusters


## 4. Cluster the features and name each cluster

Following Ng-Jordan-Weiss, the first `cluster_count` eigenvectors give each feature a new representation. Each row is normalized to unit length and **k-means** groups the features.

Each cluster is named by its **medoid**: the feature with the highest total similarity to the other features in its cluster. Cluster sizes are checked here. Many clusters of 1 to 2 features would suggest that `K_MAX` is too high for this product type.

In [71]:
embedding = eigenvectors[:, :cluster_count]
embedding = embedding / np.maximum(np.linalg.norm(embedding, axis=1, keepdims=True), 1e-12)

labels = KMeans(n_clusters=cluster_count, n_init=10, random_state=SEED).fit_predict(embedding)

results = pd.DataFrame({"Attribute": attributes, "Cluster": labels})
cluster_counts = results["Cluster"].value_counts().sort_index()
print("Cluster sizes:")
print(cluster_counts.to_string())

def group_name(label):
    indices = np.where(labels == label)[0]
    within_cluster = S[np.ix_(indices, indices)]
    return attributes[indices[within_cluster.sum(axis=1).argmax()]]

cluster_names = {l: group_name(l) for l in sorted(results["Cluster"].unique())}
results["Cluster name"] = results["Cluster"].map(cluster_names)

print("\nCLUSTER NAMES")
for c in sorted(cluster_names):
    print(f"Cluster {c}: {cluster_names[c]} ({cluster_counts.loc[c]} attributes)")

display(results.sort_values(["Cluster", "Attribute"])[["Cluster", "Cluster name", "Attribute"]].reset_index(drop=True))

Cluster sizes:
Cluster
0    40
1    11
2    11

CLUSTER NAMES
Cluster 0: Race fit (40 attributes)
Cluster 1: Elastic Interface chamois (PARIS HCS) (11 attributes)
Cluster 2: DWR treatment (11 attributes)


,Cluster,Cluster name,Attribute
0,0,Race fit,1 top entry leg pocket
1,0,Race fit,Active Fit
2,0,Race fit,Adjustable leg openings
3,0,Race fit,Anatomically optimised fit
4,0,Race fit,Comfortable mesh suspenders
...,...,...,...
57,2,DWR treatment,High-performance fabric with 10K water resista...
58,2,DWR treatment,Optimised durability
59,2,DWR treatment,Reflective details
60,2,DWR treatment,Waterproof (8K) and breathable (15K)


## 6. Visualize the clusters

t-SNE projects the features into 2D using the precomputed distances (1 - similarity). It is only used for **visualization** and does not change the clusters. Perplexity is reduced from 30 to at most 15 because there are far fewer points than in the full run. The eight largest clusters are coloured, the other groups are grey and singletons are hollow. The bar chart shows the size of every cluster.

t-SNE preserves local neighbourhoods, not global distances, so only the tightness and separation of clusters should be interpreted, not how far apart they are.

In [72]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from IPython.display import HTML, display

HIGHLIGHT_GROUPS = 8
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GRAY, LIGHT_GRAY, INK = "#8c8c86", "#c9c8c0", "#1f2933"

# t-SNE makes a 2D picture from similarities; it does not change the clusters.
distance = np.clip(1 - S, 0, None)
np.fill_diagonal(distance, 0)
coordinates = TSNE(n_components=2, metric="precomputed", init="random",
                   perplexity=min(15, (n - 1) // 3), random_state=42).fit_transform(distance)
ranked = sorted(cluster_names, key=lambda label: -cluster_counts[label])
highlighted = ranked[:HIGHLIGHT_GROUPS]
figure = go.Figure()


def add_group(indices, name, color, size=7, hollow=False):
    if len(indices) == 0:
        return
    figure.add_trace(go.Scatter(
        x=coordinates[indices, 0], y=coordinates[indices, 1], mode="markers",
        name=name, text=[attributes[i] for i in indices],
        customdata=[cluster_names.get(labels[i], "Singleton") for i in indices],
        marker=dict(size=size, color="white" if hollow else color,
                    line=dict(width=1.5 if hollow else 1, color=color if hollow else "white")),
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    ))


singletons = np.where(np.array([cluster_counts[label] == 1 for label in labels]))[0]
others = np.where(np.array([
    cluster_counts[label] > 1 and label not in highlighted for label in labels
]))[0]
add_group(singletons, f"Singletons ({len(singletons)})", LIGHT_GRAY, hollow=True)
add_group(others, f"Other groups ({len(ranked) - len(highlighted)})", GRAY)
for color, label in zip(PALETTE, highlighted):
    add_group(np.where(labels == label)[0],
              f"{cluster_names[label]} ({cluster_counts[label]})", color, size=10)

figure.update_layout(
    title=dict(text=f"Attribute clusters: {CATEGORY} (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white", width=700, height=450,
    margin=dict(l=15, r=210, t=45, b=15),
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK, size=10), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))
